# Absent Edges, Exp A: reproduce the tables on a CPU

This notebook reads the dataset package **`absent-edges-expA-v1`** (attach it as a Kaggle Dataset,
<https://www.kaggle.com/datasets/rarjunthilak/absent-edges-expa-v1>; it is expected at `/kaggle/input/absent-edges-expa-v1/`, and otherwise the first cell looks for
`reproduce/verify.py` up to four levels below `/kaggle/input/`) and

1. runs the package's verifier `reproduce/verify.py`. The verifier re-runs the frozen analyses from the released scores
   and checks every hash chain down to the episode records;
2. prints the main tables of every run: the preregistered contrasts H1 to H3, the recovery rates by condition, and the
   pre-specified cross-run comparisons.

It needs no GPU, no internet access and no extra packages (standard library only). It writes only to
`/kaggle/working/`. The code and the paper are in the companion GitHub repository; this notebook is
`notebooks/reproduce_tables.ipynb` there.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

EXPECTED = Path("/kaggle/input/absent-edges-expa-v1")


def find_dataset_root() -> Path:
    # The dataset may be mounted at the expected path or one or two levels deeper.
    if (EXPECTED / "reproduce" / "verify.py").is_file():
        return EXPECTED
    base = Path("/kaggle/input")
    for depth in range(1, 5):
        for candidate in sorted(base.glob("/".join(["*"] * depth) + "/reproduce/verify.py")):
            return candidate.parents[1]
    raise FileNotFoundError("No reproduce/verify.py under /kaggle/input: attach the dataset absent-edges-expA-v1.")


DATA = find_dataset_root()
WORK = Path("/kaggle/working/absent-edges-verify")
WORK.mkdir(parents=True, exist_ok=True)
print("dataset root:", DATA)
print("python:", sys.version.split()[0])
for entry in sorted(DATA.iterdir()):
    print(("dir   " if entry.is_dir() else "file  ") + entry.name)

## 1. Run the verifier

The verifier checks every file against the pinned hashes and re-reads every recorded witness; streams every episode
record and checks the score → result → events chain; re-runs each run's frozen analysis with the released code and
compares `CELLS.jsonl`, `TABLES.md` and `ANALYSIS.json` byte for byte; and re-runs the independent re-analysis and the
cross-run comparisons. It does not re-score episodes from raw events (that needs the upstream source trees). The first
cell prints its options (the dataset card describes them); the second runs it once and ends with a line
`SUMMARY n/n checks passed`. It takes a few minutes on a 4-CPU machine.

In [ ]:
VERIFY = DATA / "reproduce" / "verify.py"
subprocess.run([sys.executable, "-I", "-B", str(VERIFY), "--help"], cwd=WORK, check=False)

In [ ]:
VERIFY_ARGS = ["--root", str(DATA), "--jobs", "4", "--report", str(WORK / "verify.json")]
start = time.time()
proc = subprocess.run([sys.executable, "-I", "-B", str(VERIFY), *VERIFY_ARGS], cwd=WORK,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(f"===== verify.py: exit code {proc.returncode} ({time.time() - start:.0f} s)")
print(proc.stdout[-20000:])

## 2. Main tables

The cells below read the frozen `ANALYSIS.json`, `TABLES.md` and `COMPARE.json` files of the package, which the
verifier has just reproduced (it compares its regenerated copies in a temporary directory and keeps only its report,
`/kaggle/working/absent-edges-verify/verify.json`). All numbers are printed from those files; none is typed here.

In [ ]:
def frozen_files(name):
    return [p for p in sorted(DATA.rglob(name)) if p.is_file()]


def pct(x):
    return "n/a" if x is None else f"{100 * x:+.1f}"


rows = []
for path in frozen_files("ANALYSIS.json"):
    try:
        doc = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        print("skipped", path, error)
        continue
    if not str(doc.get("schema_version", "")).startswith("expA-analysis"):
        continue
    population = doc.get("population", {})
    for h in doc.get("primary", {}).get("hypotheses", []):
        interval = (h.get("t_test") or {}).get("interval") or {}
        rows.append((str(path.relative_to(DATA) if path.is_relative_to(DATA) else path), doc.get("label"), h.get("id"),
                     f"{h.get('treatment')} - {h.get('control')}", pct(h.get("estimate")),
                     f"[{pct(interval.get('lower'))}, {pct(interval.get('upper'))}]",
                     h.get("holm_adjusted_p"), h.get("status"), population.get("n_families"), population.get("n_items")))

print("Primary contrasts (percentage points; 95% t interval over library means; Holm-adjusted p)")
print(f"{'file':72} {'label':24} {'H':3} {'contrast':10} {'est':>7} {'95% CI':>17} {'Holm p':>10} {'status':13} fam items")
for r in rows:
    holm = "n/a" if r[6] is None else f"{r[6]:.2g}"
    print(f"{r[0]:72} {str(r[1]):24} {r[2]:3} {r[3]:10} {r[4]:>7} {r[5]:>17} {holm:>10} {str(r[7]):13} {r[8]} {r[9]}")

In [ ]:
def section(text, title):
    out, on = [], False
    for line in text.splitlines():
        if line.startswith("## "):
            on = line[3:].strip().startswith(title)
        if on:
            out.append(line)
    return "\n".join(out).strip()


for path in frozen_files("TABLES.md"):
    text = path.read_text(encoding="utf-8")
    if "expA-analysis" not in text[:200]:
        continue
    print("=" * 100)
    print(path.relative_to(DATA) if path.is_relative_to(DATA) else path)
    print(text.splitlines()[2] if len(text.splitlines()) > 2 else "")
    for title in ("Primary hypotheses", "Rates by block and condition"):
        print()
        print(section(text, title) or f"(no section '{title}')")

In [ ]:
for path in frozen_files("COMPARE.json"):
    doc = json.loads(path.read_text(encoding="utf-8"))
    print("=" * 100)
    print(path.relative_to(DATA) if path.is_relative_to(DATA) else path, "| mode:", doc.get("mode"))
    print(doc.get("status"))
    deltas = doc.get("delta") or doc.get("did") or {}
    for h, d in sorted(deltas.items()):
        if isinstance(d, dict) and "estimate" in d:
            print(f"  {h}: {pct(d.get('estimate'))} points, 95% CI [{pct(d.get('ci_low'))}, {pct(d.get('ci_high'))}]"
                  f", {d.get('n_families')} families")
        else:
            print(f"  {h}: {json.dumps(d)[:300]}")
    if doc.get("classification"):
        print("  classification:", doc["classification"])

## Notes
- Episode reruns need a GPU and the model weights; they are not part of this notebook (see `docs/EXPA_RUNBOOK.md` in
  the code repository).
- `reproduce/verify.py` refuses changed inputs. A FAIL line names the file and the pin it broke.
- The paper's own number check is in the code repository (`paper/tools/claims.py verify`).